In [0]:
#config/gold/marketo_lead_folloze_mapping.yaml

In [0]:
dbutils.widgets.text("mounting_point","")
dbutils.widgets.text("yaml_path", "")

In [0]:
mounting_point= dbutils.widgets.get("mounting_point")
yaml_path = dbutils.widgets.get("yaml_path")

In [0]:
%sql
create database if not exists silver;
create database if not exists gold;

In [0]:
spark.conf.set(
  "fs.azure.account.key.gdmdatalakep.dfs.core.windows.net",
  dbutils.secrets.get(scope="databricks-secret-scope",key="adlsstoragekey"))

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false

In [0]:
from delta import DeltaTable
from delta.tables import *
import json
import yaml
from pyspark.sql.functions import *
from pyspark.sql.types import *
import databricks.koalas as ks

with open(f'/dbfs/{mounting_point}/{yaml_path}') as f:
    data = yaml.load(f, Loader=yaml.FullLoader)
for data in data:
    source_container = data['source-storage-container']
    source_storage_path = data['source-storage-path']
    source_storage_account = data['source-storage-account']
    source_data_type =data['source-data-type']
    source_data_ext =data['source-data-extn']
    try:
      dest_file = data['dest-file-name']
    except:
      dest_file = ""
    try:
      dest_table = data['dest-table']
    except:
      dest_table = ""
    try:
      dest_table_dest = data['dest-table-path']
    except:
      dest_table_dest = ""
      
    container = data['dest-storage-container']
    account = data['dest-storage-account']
    path = data['dest-storage-path']
    zone = data['zone']
    query_type = data.get('query-type')
    add_file_date = data.get('add-file-date-col')
    
    try:
      json_column = data['json_column']
    except:
      json_column = ""
    try:
      where_clause = data['where']
      where_clause = (" and ".join(where_clause))
    except:
      where_clause = ""
      
    try:
      merge_key = data['merge-columns']
      #construct the mergekey
      merge_key = [f'source.{item} <=> updates.{item}'  for item in merge_key]
      merge_key = (" and ".join(merge_key))  
    except:
      merge_key = ""
      
    try:
      partitions = data['partition-col']
    except:
      partitions = ""
    try:
      select_array = data['select-column-def']
    except:
      select_array = ""
    try:
      schema_loc = data['schema']
    except:
      schema_loc = ""

    if schema_loc != "":
      schema_df = spark.read.json(f'dbfs://{mounting_point}/{schema_loc}',multiLine=True)
      

    #set saprk to be case sensitive if required
    if(data.get('case-senstive')==True):
      spark.sql("set spark.sql.caseSensitive=true")
## go and get source data to merge based on yaml definition
#ansure that parquet and csv are supported
#reges to replace special characters
    if query_type != None and query_type == "hive":
      selected_table = spark.sql(data.get("select-expression"))
    elif source_data_type == "json":
      selected_table = spark.read.json(f'abfss://{source_container}@{source_storage_account}.dfs.core.windows.net/{source_storage_path}/*.{source_data_ext}')
    elif source_data_type == "json_with_schema":
      selected_table = spark.read.json(f'abfss://{source_container}@{source_storage_account}.dfs.core.windows.net/{source_storage_path}/*.{source_data_ext}', schema_df.schema)
    elif source_data_type == "delta":
      selected_table = spark.read.format("delta").load(f'abfss://{source_container}@{source_storage_account}.dfs.core.windows.net/{source_storage_path}/')
    elif source_data_type == "parquet":
      selected_table = spark.read.parquet(f'abfss://{source_container}@{source_storage_account}.dfs.core.windows.net/{source_storage_path}/*.{source_data_ext}')
    elif source_data_type == "csv":
      selected_table = spark.read.csv(f'abfss://{source_container}@{source_storage_account}.dfs.core.windows.net/{source_storage_path}/*.{source_data_ext}', header=True, escape = "\"", quote = "\"", multiLine=True, nullValue = 'null')
    elif source_data_type == "csv_with_json_column":
      selected_table = spark.read.csv(f'abfss://{source_container}@{source_storage_account}.dfs.core.windows.net/{source_storage_path}/*.{source_data_ext}', header=True, escape = "\"", nullValue = 'null')\
                            .withColumn("json_data", from_json(col(json_column), MapType(StringType(), StringType())))
#     elif source_data_type == "xlsx":
#       selected_table = ks.read_excel(f'abfss://{source_container}@{source_storage_account}.dfs.core.windows.net/{source_storage_path}/*.{source_data_ext}',sheet_name=data.get('sheet-name'),engine='openpyxl')
#       selected_table = selected_table.to_spark()
    else:
      selected_table = spark.read.json(f'abfss://{source_container}@{source_storage_account}.dfs.core.windows.net/{source_storage_path}/*.{source_data_ext}')
      
    #add File Name, file date and process date for all landing file 
    if query_type == None:
      selected_table = selected_table.withColumn("input_file_name", reverse(split(input_file_name(),'/'))[0])
      selected_table = selected_table.withColumn('file_extract_date',to_timestamp(concat(regexp_extract(col("input_file_name"), r'(\w+)_(\d+)_(\d+).(\w+)', 2)\
                                                                                       ,regexp_extract(col("input_file_name"), r'(\w+)_(\d+)_(\d+).(\w+)', 3)).cast("string"), "yyyyMMddHHmmss"))
    source_record_count = selected_table.count() 
    
    if select_array != "" and select_array != None:
      for x in select_array:
        selected_table = selected_table.selectExpr(x['select'])
    else:
      new_column_names = [f"{c.replace(' ', '_').lower().replace('(','').replace(')','').replace('.','')}" for c in selected_table.columns]
      selected_table = selected_table.toDF(*new_column_names)
       
    if where_clause != "":
      selected_table = selected_table.where(where_clause)
  
#merge data into dest
    target_record_count = selected_table.count()
    delta_path = f'abfss://{container}@{account}.dfs.core.windows.net/{path}/{dest_table_dest}' 
    exists = DeltaTable.isDeltaTable(spark,delta_path)

    if exists == False:
      if partitions != "":
        selected_table.write.partitionBy(*partitions).format("delta").save(delta_path)
      else:
        selected_table.write.format("delta").save(delta_path)
      spark.sql(f"CREATE TABLE IF NOT EXISTS {zone}.{dest_table} USING DELTA LOCATION '{delta_path}'")
    else:
      if partitions != "":
        #modify for merge
        selected_table.write.partitionBy(*partitions).format("delta").option("mergeSchema", "true").mode("overwrite").save(delta_path)
      else:
        if merge_key != "":
          deltaTableDest = DeltaTable.forPath(spark,delta_path)
          deltaTableDest.alias("source").merge(
              selected_table.alias("updates"),
              merge_key) \
          .whenMatchedUpdateAll() \
          .whenNotMatchedInsertAll().execute()
        else:
          selected_table.write.format("delta").option("mergeSchema", "true").mode("overwrite").save(delta_path)
      spark.sql(f"CREATE TABLE IF NOT EXISTS {zone}.{dest_table} USING DELTA LOCATION '{delta_path}'")
      
      deltaTableDest = DeltaTable.forPath(spark,delta_path)
      deltaTableDest.vacuum(0.000001)
      print('finished vacumming')

#set saprk to be case insensitive if set to sensitive earlier in notebook     
if(data.get('case-senstive')==True):
  spark.sql("set spark.sql.caseSensitive=false")

In [0]:
# Output Delta table, source count and target count information
dbutils.notebook.exit([container, dest_table_dest, source_record_count, target_record_count])